In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

# =========================================================================
# Hyperparameters & Paths
# =========================================================================
BASE_DIR = "./ecg_raw_128dim_dataset"
BATCH_SIZE = 512
EPOCHS = 35
LEARNING_RATE = 2e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ['class_0_normal', 'class_1_arrhythmia', 'class_2_chf']

# Multiple CS ratios to benchmark: 75% (32), 50% (64), 25% (96), 0% uncompressed (128)
CS_DIMS_TO_EVALUATE = [32, 64, 96, 128]

# =========================================================================
# 1. Dataset Loader with Dynamic On-the-Fly Projection Matrix
# =========================================================================
class ECGMultiModalDataset(Dataset):
    def __init__(self, base_dir, split='train', m_dim=32, stats=None, seed=42):
        self.m_dim = m_dim
        self.raw_beats, self.rr_data, self.hrv_data, self.labels = [], [], [], []

        for label_idx, cls_name in enumerate(CLASS_NAMES):
            cls_dir = os.path.join(base_dir, split, cls_name)
            raw = np.load(os.path.join(cls_dir, 'beats_raw.npy')).astype(np.float32)
            rr = np.load(os.path.join(cls_dir, 'beats_rr.npy')).astype(np.float32)
            hrv = np.load(os.path.join(cls_dir, 'beats_hrv.npy')).astype(np.float32)

            min_len = min(len(raw), len(rr), len(hrv))
            self.raw_beats.append(raw[:min_len])
            self.rr_data.append(rr[:min_len])
            self.hrv_data.append(hrv[:min_len])
            self.labels.append(np.full((min_len,), label_idx, dtype=np.int64))

        self.raw_beats = np.concatenate(self.raw_beats, axis=0)
        self.rr_data = np.concatenate(self.rr_data, axis=0)
        self.hrv_data = np.concatenate(self.hrv_data, axis=0)
        self.labels = np.concatenate(self.labels, axis=0)

        # Standardize tabular features on training set statistics
        if stats is None:
            self.rr_mean = np.mean(self.rr_data, axis=0)
            self.rr_std = np.std(self.rr_data, axis=0) + 1e-8
            self.hrv_mean = np.mean(self.hrv_data, axis=0)
            self.hrv_std = np.std(self.hrv_data, axis=0) + 1e-8
        else:
            self.rr_mean, self.rr_std, self.hrv_mean, self.hrv_std = stats

        self.rr_data = (self.rr_data - self.rr_mean) / self.rr_std
        self.hrv_data = (self.hrv_data - self.hrv_mean) / self.hrv_std

        # Construct Gaussian Sensing Matrix Phi ~ N(0, 1/M)
        if self.m_dim < 128:
            rng = np.random.RandomState(seed)
            self.phi = rng.randn(self.m_dim, 128).astype(np.float32) * np.sqrt(1.0 / self.m_dim)
            # Project 128-dim raw ECG down to M-dim CS space: y = x @ Phi.T
            self.cs_data = np.dot(self.raw_beats, self.phi.T)
        else:
            self.cs_data = self.raw_beats

    def get_stats(self):
        return (self.rr_mean, self.rr_std, self.hrv_mean, self.hrv_std)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.cs_data[idx], dtype=torch.float32),
            torch.tensor(self.rr_data[idx], dtype=torch.float32),
            torch.tensor(self.hrv_data[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# =========================================================================
# 2. Flexible Model (Supports Dual-Branch CS+RR or Tri-Branch CS+RR+HRV)
# =========================================================================
class ModularECGClassifier(nn.Module):
    def __init__(self, m_dim=32, num_classes=3, use_hrv=True):
        super(ModularECGClassifier, self).__init__()
        self.use_hrv = use_hrv

        # Branch 1: CS Morphology
        self.branch_cs = nn.Sequential(
            nn.Linear(m_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 16),
            nn.BatchNorm1d(16),
            nn.ReLU()
        )

        # Branch 2: Local Rhythm Intervals
        self.branch_rr = nn.Sequential(
            nn.Linear(4, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.BatchNorm1d(16),
            nn.ReLU()
        )

        # Branch 3: Autonomic HRV Dynamics (Optional ablation)
        if self.use_hrv:
            self.branch_hrv = nn.Sequential(
                nn.Linear(8, 16),
                nn.BatchNorm1d(16),
                nn.ReLU(),
                nn.Linear(16, 16),
                nn.BatchNorm1d(16),
                nn.ReLU()
            )
            head_in_dim = 16 + 16 + 16  # 48
        else:
            head_in_dim = 16 + 16       # 32

        # Final Classifier Head
        self.classifier_head = nn.Sequential(
            nn.Linear(head_in_dim, 24 if self.use_hrv else 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(24 if self.use_hrv else 16, num_classes)
        )

    def forward(self, x_cs, x_rr, x_hrv=None):
        f_cs = self.branch_cs(x_cs)
        f_rr = self.branch_rr(x_rr)

        if self.use_hrv and x_hrv is not None:
            f_hrv = self.branch_hrv(x_hrv)
            fused = torch.cat((f_cs, f_rr, f_hrv), dim=1)
        else:
            fused = torch.cat((f_cs, f_rr), dim=1)

        return self.classifier_head(fused)

# =========================================================================
# 3. Loss Function (Focal Loss)
# =========================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return (((1.0 - pt) ** self.gamma) * ce_loss).mean()

# =========================================================================
# 4. Training & Benchmarking Routine
# =========================================================================
def train_and_eval_configuration(m_dim, use_hrv=True):
    mode_str = "Tri-Branch (CS + RR + HRV)" if use_hrv else "Dual-Branch (CS + RR Baseline)"
    print(f"\n{'='*75}\n RUNNING: M = {m_dim}-dim | Mode = {mode_str}\n{'='*75}")

    # Initialize Datasets
    train_dataset = ECGMultiModalDataset(BASE_DIR, split='train', m_dim=m_dim)
    test_dataset = ECGMultiModalDataset(BASE_DIR, split='test', m_dim=m_dim, stats=train_dataset.get_stats())

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Compute square-root balanced weights
    labels = train_dataset.labels
    class_counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    weights = np.sqrt(len(labels) / (len(CLASS_NAMES) * class_counts.astype(np.float32)))
    weights = weights / weights.mean()
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model = ModularECGClassifier(m_dim=m_dim, num_classes=3, use_hrv=use_hrv).to(DEVICE)
    criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    # Train Loop
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for x_cs, x_rr, x_hrv, y in train_loader:
            x_cs, x_rr, x_hrv, y = x_cs.to(DEVICE), x_rr.to(DEVICE), x_hrv.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()

            logits = model(x_cs, x_rr, x_hrv if use_hrv else None)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(y)
            correct += (torch.argmax(logits, dim=1) == y).sum().item()
            total += len(y)

        scheduler.step()
        if epoch % 10 == 0 or epoch == EPOCHS:
            print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] - Loss: {running_loss/total:.4f} | Acc: {(correct/total)*100:.2f}%")

    # Evaluation Loop
    model.eval()
    all_preds, all_targets = [], []
    ARRHYTHMIA_THRESHOLD = 0.65

    with torch.no_grad():
        for x_cs, x_rr, x_hrv, y in test_loader:
            x_cs, x_rr, x_hrv = x_cs.to(DEVICE), x_rr.to(DEVICE), x_hrv.to(DEVICE)
            logits = model(x_cs, x_rr, x_hrv if use_hrv else None)
            probs = torch.softmax(logits, dim=1).cpu().numpy()

            p_norm, p_arr, p_chf = probs[:, 0], probs[:, 1], probs[:, 2]
            base_choice = np.where(p_norm >= p_chf, 0, 2)
            preds = np.where(p_arr >= ARRHYTHMIA_THRESHOLD, 1, base_choice)

            all_preds.extend(preds)
            all_targets.extend(y.numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    print("\nTest Evaluation Summary:")
    print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(all_targets, all_preds))

if __name__ == '__main__':
    # 1. Run the CS Baseline across compression ratios (CS + RR only)
    print(">>> SECTION 1: BENCHMARKING COMPRESSIVE SENSING RATIOS (DUAL-BRANCH)")
    for m in CS_DIMS_TO_EVALUATE:
        train_and_eval_configuration(m_dim=m, use_hrv=False)

    # 2. Run the Full Fusion across compression ratios (CS + RR + Continuous HRV)
    print("\n>>> SECTION 2: BENCHMARKING AUTONOMIC FUSION (TRI-BRANCH)")
    for m in CS_DIMS_TO_EVALUATE:
        train_and_eval_configuration(m_dim=m, use_hrv=True)

>>> SECTION 1: BENCHMARKING COMPRESSIVE SENSING RATIOS (DUAL-BRANCH)

 RUNNING: M = 32-dim | Mode = Dual-Branch (CS + RR Baseline)
Epoch [10/35] - Loss: 0.0147 | Acc: 96.30%
Epoch [20/35] - Loss: 0.0117 | Acc: 96.67%
Epoch [30/35] - Loss: 0.0097 | Acc: 97.13%
Epoch [35/35] - Loss: 0.0098 | Acc: 97.17%

Test Evaluation Summary:
                    precision    recall  f1-score   support

    class_0_normal     0.8941    0.9209    0.9073     64324
class_1_arrhythmia     0.4439    0.7408    0.5552      5506
       class_2_chf     0.9497    0.7828    0.8582     31906

          accuracy                         0.8678    101736
         macro avg     0.7626    0.8148    0.7736    101736
      weighted avg     0.8872    0.8678    0.8729    101736

Confusion Matrix:
[[59236  4069  1019]
 [ 1124  4079   303]
 [ 5891  1040 24975]]

 RUNNING: M = 64-dim | Mode = Dual-Branch (CS + RR Baseline)
Epoch [10/35] - Loss: 0.0130 | Acc: 96.50%
Epoch [20/35] - Loss: 0.0099 | Acc: 97.23%
Epoch [30/35] - Lo

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

BASE_DIR = "./ecg_raw_128dim_dataset"
BATCH_SIZE = 512
EPOCHS = 35
LEARNING_RATE = 2e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ['class_0_normal', 'class_1_arrhythmia', 'class_2_chf']
CS_DIMS_TO_EVALUATE = [32, 64, 96, 128]

# =========================================================================
# 1. Dataset Loader with Gaussian Sensing Matrix Projection
# =========================================================================
class ECGMultiModalDataset(Dataset):
    def __init__(self, base_dir, split='train', m_dim=32, stats=None, seed=42):
        self.m_dim = m_dim
        self.raw_beats, self.rr_data, self.hrv_data, self.labels = [], [], [], []

        for label_idx, cls_name in enumerate(CLASS_NAMES):
            cls_dir = os.path.join(base_dir, split, cls_name)
            raw = np.load(os.path.join(cls_dir, 'beats_raw.npy')).astype(np.float32)
            rr = np.load(os.path.join(cls_dir, 'beats_rr.npy')).astype(np.float32)
            hrv = np.load(os.path.join(cls_dir, 'beats_hrv.npy')).astype(np.float32)

            min_len = min(len(raw), len(rr), len(hrv))
            self.raw_beats.append(raw[:min_len])
            self.rr_data.append(rr[:min_len])
            self.hrv_data.append(hrv[:min_len])
            self.labels.append(np.full((min_len,), label_idx, dtype=np.int64))

        self.raw_beats = np.concatenate(self.raw_beats, axis=0)
        self.rr_data = np.concatenate(self.rr_data, axis=0)
        self.hrv_data = np.concatenate(self.hrv_data, axis=0)
        self.labels = np.concatenate(self.labels, axis=0)

        # Standardize tabular features
        if stats is None:
            self.rr_mean = np.mean(self.rr_data, axis=0)
            self.rr_std = np.std(self.rr_data, axis=0) + 1e-8
            self.hrv_mean = np.mean(self.hrv_data, axis=0)
            self.hrv_std = np.std(self.hrv_data, axis=0) + 1e-8
        else:
            self.rr_mean, self.rr_std, self.hrv_mean, self.hrv_std = stats

        self.rr_data = (self.rr_data - self.rr_mean) / self.rr_std
        self.hrv_data = (self.hrv_data - self.hrv_mean) / self.hrv_std

        # Gaussian sensing matrix Phi ~ N(0, 1/M)
        if self.m_dim < 128:
            rng = np.random.RandomState(seed)
            self.phi = rng.randn(self.m_dim, 128).astype(np.float32) * np.sqrt(1.0 / self.m_dim)
            self.cs_data = np.dot(self.raw_beats, self.phi.T)
        else:
            self.cs_data = self.raw_beats

    def get_stats(self):
        return (self.rr_mean, self.rr_std, self.hrv_mean, self.hrv_std)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.cs_data[idx], dtype=torch.float32),
            torch.tensor(self.rr_data[idx], dtype=torch.float32),
            torch.tensor(self.hrv_data[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# =========================================================================
# 2. Asymmetric Architecture with 1D-CNN Stem & Modality Dropout
# =========================================================================
class AsymmetricECGClassifier(nn.Module):
    def __init__(self, m_dim=32, num_classes=3, use_hrv=True, hrv_dropout_p=0.40):
        super(AsymmetricECGClassifier, self).__init__()
        self.use_hrv = use_hrv
        self.hrv_dropout_p = hrv_dropout_p

        # Branch 1: 1D-CNN Stem for Compressed Waveform (Output: 32 dimensions)
        self.branch_cs = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(output_size=4),  # Normalizes arbitrary M length to fixed (32 x 4)
            nn.Flatten(),
            nn.Linear(32 * 4, 32),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        # Branch 2: Local Rhythm Intervals (Output: 8 dimensions)
        self.branch_rr = nn.Sequential(
            nn.Linear(4, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.BatchNorm1d(8),
            nn.ReLU()
        )

        # Branch 3: Autonomic HRV Dynamics (Output: 8 dimensions)
        if self.use_hrv:
            self.branch_hrv = nn.Sequential(
                nn.Linear(8, 16),
                nn.BatchNorm1d(16),
                nn.ReLU(),
                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.ReLU()
            )
            head_in_dim = 32 + 8 + 8  # 48 total, dominated by CS (32)
        else:
            head_in_dim = 32 + 8      # 40 total

        self.classifier_head = nn.Sequential(
            nn.Linear(head_in_dim, 24),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(24, num_classes)
        )

    def forward(self, x_cs, x_rr, x_hrv=None):
        # Reshape (Batch, M) -> (Batch, 1, M) for Conv1d operations
        x_cs_1d = x_cs.unsqueeze(1)
        f_cs = self.branch_cs(x_cs_1d)
        f_rr = self.branch_rr(x_rr)

        if self.use_hrv and x_hrv is not None:
            f_hrv = self.branch_hrv(x_hrv)
            
            # Modality Dropout: Stochastic masking of the entire HRV branch during training
            if self.training and self.hrv_dropout_p > 0.0:
                mask = (torch.rand(f_hrv.shape[0], 1, device=f_hrv.device) >= self.hrv_dropout_p).float()
                f_hrv = f_hrv * mask

            fused = torch.cat((f_cs, f_rr, f_hrv), dim=1)
        else:
            fused = torch.cat((f_cs, f_rr), dim=1)

        return self.classifier_head(fused)

# =========================================================================
# 3. Training & Evaluation Engine
# =========================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return (((1.0 - pt) ** self.gamma) * ce_loss).mean()

def train_and_eval_configuration(m_dim, use_hrv=True):
    mode_str = "Tri-Branch (1D-CNN Asymmetric + Modality Dropout)" if use_hrv else "Dual-Branch (1D-CNN Baseline)"
    print(f"\n{'='*75}\n RUNNING: M = {m_dim}-dim | Mode = {mode_str}\n{'='*75}")

    train_dataset = ECGMultiModalDataset(BASE_DIR, split='train', m_dim=m_dim)
    test_dataset = ECGMultiModalDataset(BASE_DIR, split='test', m_dim=m_dim, stats=train_dataset.get_stats())

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    labels = train_dataset.labels
    class_counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    weights = np.sqrt(len(labels) / (len(CLASS_NAMES) * class_counts.astype(np.float32)))
    weights = weights / weights.mean()
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

    model = AsymmetricECGClassifier(m_dim=m_dim, num_classes=3, use_hrv=use_hrv, hrv_dropout_p=0.40).to(DEVICE)
    criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for x_cs, x_rr, x_hrv, y in train_loader:
            x_cs, x_rr, x_hrv, y = x_cs.to(DEVICE), x_rr.to(DEVICE), x_hrv.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()

            logits = model(x_cs, x_rr, x_hrv if use_hrv else None)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(y)
            correct += (torch.argmax(logits, dim=1) == y).sum().item()
            total += len(y)

        scheduler.step()
        if epoch % 10 == 0 or epoch == EPOCHS:
            print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] - Loss: {running_loss/total:.4f} | Acc: {(correct/total)*100:.2f}%")

    # Evaluation
    model.eval()
    all_preds, all_targets = [], []
    ARRHYTHMIA_THRESHOLD = 0.65

    with torch.no_grad():
        for x_cs, x_rr, x_hrv, y in test_loader:
            x_cs, x_rr, x_hrv = x_cs.to(DEVICE), x_rr.to(DEVICE), x_hrv.to(DEVICE)
            logits = model(x_cs, x_rr, x_hrv if use_hrv else None)
            probs = torch.softmax(logits, dim=1).cpu().numpy()

            p_norm, p_arr, p_chf = probs[:, 0], probs[:, 1], probs[:, 2]
            base_choice = np.where(p_norm >= p_chf, 0, 2)
            preds = np.where(p_arr >= ARRHYTHMIA_THRESHOLD, 1, base_choice)

            all_preds.extend(preds)
            all_targets.extend(y.numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    print("\nTest Evaluation Summary:")
    print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(all_targets, all_preds))

if __name__ == '__main__':
    print(">>> SECTION 1: BENCHMARKING 1D-CNN CS BASELINE (DUAL-BRANCH)")
    for m in CS_DIMS_TO_EVALUATE:
        train_and_eval_configuration(m_dim=m, use_hrv=False)

    print("\n>>> SECTION 2: BENCHMARKING ASYMMETRIC FUSION WITH MODALITY DROPOUT (TRI-BRANCH)")
    for m in CS_DIMS_TO_EVALUATE:
        train_and_eval_configuration(m_dim=m, use_hrv=True)

>>> SECTION 1: BENCHMARKING 1D-CNN CS BASELINE (DUAL-BRANCH)

 RUNNING: M = 32-dim | Mode = Dual-Branch (1D-CNN Baseline)
Epoch [10/35] - Loss: 0.0099 | Acc: 97.62%
Epoch [20/35] - Loss: 0.0060 | Acc: 98.43%
Epoch [30/35] - Loss: 0.0036 | Acc: 99.04%
Epoch [35/35] - Loss: 0.0033 | Acc: 99.13%

Test Evaluation Summary:
                    precision    recall  f1-score   support

    class_0_normal     0.8929    0.8919    0.8924     64324
class_1_arrhythmia     0.4015    0.7268    0.5173      5506
       class_2_chf     0.9066    0.7818    0.8396     31906

          accuracy                         0.8485    101736
         macro avg     0.7337    0.8002    0.7498    101736
      weighted avg     0.8706    0.8485    0.8556    101736

Confusion Matrix:
[[57373  4679  2272]
 [ 1205  4002   299]
 [ 5674  1287 24945]]

 RUNNING: M = 64-dim | Mode = Dual-Branch (1D-CNN Baseline)
Epoch [10/35] - Loss: 0.0090 | Acc: 97.70%
Epoch [20/35] - Loss: 0.0058 | Acc: 98.53%
Epoch [30/35] - Loss: 0.0033